# Binding-site prediction — survey of off-the-shelf methods

Second approach to the challenge: instead of training a new model, we plug
every test protein through several established binding-site predictors and
turn each output into the submission format expected by `metric.py`.

Methods covered:

1. **P2Rank** — pre-trained random-forest scoring of SAS points (Java).
2. **Fpocket** — purely geometric alpha-sphere detection.
3. **DeepSite** — 3D-CNN binding-site predictor served by PlayMolecule.
4. **DeepSurf** — 3D-CNN over surface point clouds (TensorFlow).
5. **Custom 3D voxel U-Net** — small segmentation network, reference only.

Local Jaccard on a 200-protein train subset (matches `metric.py`):

| method                       | Jaccard |
| ---------------------------- | ------- |
| P2Rank top-1 pocket          | 0.263   |
| **P2Rank top-2 pockets**     | **0.282** |
| P2Rank top-3 pockets         | 0.269   |
| P2Rank per-residue p ≥ 0.30  | 0.221   |
| P2Rank per-residue p ≥ 0.40  | 0.207   |
| P2Rank per-residue p ≥ 0.50  | 0.190   |
| Fpocket top-1                | 0.143   |
| Fpocket top-2                | 0.172   |
| Fpocket top-3                | 0.178   |
| Fpocket top-5                | 0.182   |
| Union(P2Rank top-2, Fpocket top-2)     | 0.268   |
| Intersection(P2Rank top-2, Fpocket top-2) | 0.163   |

Best local score is **P2Rank top-2 pockets**, so that is the submission
written to `02_offshelf_tools/submission.csv` (public LB **0.268**).

Paths: data lives under `data/` (train/, test/, csvs); heavy regenerable
artifacts (downloaded P2Rank binary, per-protein tool dumps) go to
`02_offshelf_tools/_generated/` and are gitignored. Run this notebook from
the challenge root.

## 0  Setup

In [ ]:
import os, sys, shutil, zipfile, subprocess, time
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
from sklearn.metrics import jaccard_score

ROOT = Path('.').resolve()
if ROOT.name == '02_offshelf_tools':
    ROOT = ROOT.parent
os.chdir(ROOT)
print('working dir:', ROOT)

TRAIN_CSV   = ROOT / 'data' / 'train.csv'
SAMPLE_SUB  = ROOT / 'data' / 'sample_submission.csv'

APPROACH = ROOT / '02_offshelf_tools'
GEN      = APPROACH / '_generated'          # heavy regenerable artifacts (gitignored)
TOOLS    = GEN / 'tools'                     # downloaded P2Rank binary
OUT      = GEN                               # p2rank_full/, fpocket_full/, *.ds, variant CSVs
TEST_PDB = ROOT / 'data' / 'test'            # all 3064 test PDBs (already extracted)
TRAIN_PDB_SUB = GEN / 'data' / 'train_pdb_subset'
for p in (TOOLS, OUT, TRAIN_PDB_SUB):
    p.mkdir(parents=True, exist_ok=True)

N_TEST = 3064
N_TRAIN_EVAL = 200    # how many train proteins we use for local scoring

In [ ]:
# Test PDBs are already extracted under data/test.
print('test PDBs available :', len(list(TEST_PDB.glob('*_protein.pdb'))))

# Build a small train subset for local scoring by copying from data/train
# (falls back to train.zip if only the archive is present).
TRAIN_DIR = ROOT / 'data' / 'train'
TRAIN_ZIP = ROOT / 'train.zip'
have = len(list(TRAIN_PDB_SUB.glob('*_protein.pdb')))
if have < N_TRAIN_EVAL:
    if TRAIN_DIR.is_dir():
        for pid in range(N_TRAIN_EVAL):
            src = TRAIN_DIR / f'{pid}_protein.pdb'
            if src.exists():
                shutil.copy(src, TRAIN_PDB_SUB / src.name)
    elif TRAIN_ZIP.exists():
        with zipfile.ZipFile(TRAIN_ZIP) as zf:
            names = set(zf.namelist())
            for pid in range(N_TRAIN_EVAL):
                n = f'{pid}_protein.pdb'
                if n in names:
                    zf.extract(n, TRAIN_PDB_SUB)
print('train PDB subset    :', len(list(TRAIN_PDB_SUB.glob('*_protein.pdb'))))

### Helpers shared by every method

`parse_submission` is taken verbatim from the provided `metric.py` so we
are sure every output round-trips through the same parser the leaderboard
uses.

In [ ]:
def parse_submission(df_submission):
    df_submission = df_submission.copy()
    df_submission['prediction'] = df_submission['prediction'].fillna('')
    rows = []
    for _, r in df_submission.iterrows():
        for res in str(r['prediction']).strip().split(' '):
            if res.strip():
                rows.append((r['id'], res.strip()))
    out = pd.DataFrame(rows, columns=['id', 'residue_id'])
    out['prediction'] = 1
    return out


def check_submission(path):
    df = pd.read_csv(path)
    n = len(df)
    empty = (df['prediction'].fillna('').str.strip() == '').sum()
    parsed = parse_submission(df)
    print(f'  {Path(path).name:35s} rows={n}  unique_ids={df.id.nunique():>4}  '
          f'empty={empty:>3}  tokens={len(parsed):>6}')


def residue_universe(pdb_path):
    """Residues with a Cα atom, in the chain_resnum[_icode] format."""
    out, seen = [], set()
    with open(pdb_path) as f:
        for line in f:
            if not line.startswith('ATOM') or line[12:16].strip() != 'CA':
                continue
            chain = line[21:22]
            try: num = int(line[22:26])
            except ValueError: continue
            ic = line[26:27].strip()
            tag = f'{chain}_{num}_{ic}' if ic else f'{chain}_{num}'
            if tag not in seen:
                seen.add(tag); out.append(tag)
    return out


def jaccard_local(pids, pdb_root, label_map, predict_fn):
    y_true, y_pred = [], []
    for pid in pids:
        univ  = residue_universe(Path(pdb_root) / f'{pid}_protein.pdb')
        truth = label_map.get(int(pid), set())
        preds = predict_fn(int(pid))
        for tok in univ:
            y_true.append(1 if tok in truth else 0)
            y_pred.append(1 if tok in preds else 0)
    return jaccard_score(y_true, y_pred)


check_submission(SAMPLE_SUB)

## 1  P2Rank (pre-trained random forest)

P2Rank scores solvent-accessible-surface (SAS) points with a pre-trained
random forest, then clusters high-scoring points into pockets. Each pocket
comes with the residues lining it, in the same `chain_resnum` format we
need.

We use the bundled `default.model` (no MSAs, no AlphaFold).
Reference: Krivak & Hoksza, *J. Cheminform.* (2018).

In [ ]:
P2RANK_DIR = TOOLS / 'p2rank_2.4.2'
P2RANK_BIN = P2RANK_DIR / 'prank'
P2RANK_URL = ('https://github.com/rdk/p2rank/releases/download/'
              '2.4.2/p2rank_2.4.2.tar.gz')

if not P2RANK_BIN.exists():
    tar = TOOLS / 'p2rank.tar.gz'
    subprocess.run(['curl', '-L', '-o', str(tar), P2RANK_URL], check=True)
    subprocess.run(['tar', '-xzf', str(tar), '-C', str(TOOLS)], check=True)
    tar.unlink()
    P2RANK_BIN.chmod(0o755)

print(P2RANK_BIN, 'present:', P2RANK_BIN.exists())

In [ ]:
def build_dataset_file(pdb_root, out_path):
    pdbs = sorted(Path(pdb_root).glob('*_protein.pdb'),
                  key=lambda p: int(p.stem.split('_')[0]))
    with open(out_path, 'w') as f:
        for p in pdbs:
            f.write(str(p.resolve()) + '\n')
    return len(pdbs)


def run_p2rank(pdb_root, out_dir, n_expected, threads=8):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    have = len(list(out_dir.glob('*_predictions.csv')))
    if have >= n_expected:
        print(f'  {out_dir.name}: already have {have} predictions, skipping')
        return
    ds = OUT / f'{out_dir.name}.ds'
    build_dataset_file(pdb_root, ds)
    cmd = [str(P2RANK_BIN), 'predict', str(ds),
           '-o', str(out_dir), '-threads', str(threads), '-visualizations', '0']
    print('  ', ' '.join(cmd))
    subprocess.run(cmd, check=True)

P2RANK_TEST  = OUT / 'p2rank_full'
P2RANK_TRAIN = OUT / 'p2rank_train_eval'

run_p2rank(TEST_PDB,      P2RANK_TEST,  N_TEST)
run_p2rank(TRAIN_PDB_SUB, P2RANK_TRAIN, N_TRAIN_EVAL)

csvs = sorted(P2RANK_TEST.glob('*_predictions.csv'))
print('  test predictions written:', len(csvs))
print(pd.read_csv(csvs[0]).head(3))

In [ ]:
def p2rank_topk_residues(p2rank_dir, pid, top_k=1):
    """Residue set of the K best-ranked P2Rank pockets for one protein."""
    path = Path(p2rank_dir) / f'{pid}_protein.pdb_predictions.csv'
    if not path.exists(): return set()
    df = pd.read_csv(path); df.columns = df.columns.str.strip()
    if df.empty or 'residue_ids' not in df.columns: return set()
    out = set()
    for raw in df.head(top_k)['residue_ids'].astype(str):
        for tok in raw.strip().split():
            if tok: out.add(tok)
    return out


def write_p2rank_submission(p2rank_dir, n_proteins, top_k, out_csv):
    rows = []
    for pid in range(n_proteins):
        toks = p2rank_topk_residues(p2rank_dir, pid, top_k=top_k)
        rows.append({'id': pid, 'prediction': ' '.join(sorted(toks))})
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    return out_csv

for k in (1, 2, 3):
    p = write_p2rank_submission(P2RANK_TEST, N_TEST, k,
                                OUT / f'submission_p2rank_top{k}.csv')
    check_submission(p)

In [ ]:
# Local Jaccard for every P2Rank strategy on the train subset.
labels = pd.read_csv(TRAIN_CSV)
label_map = {int(r['id']): set(str(r['resid']).split())
             for _, r in labels.iterrows()}
train_pids = [p for p in range(N_TRAIN_EVAL)
              if label_map.get(p) and
              (TRAIN_PDB_SUB / f'{p}_protein.pdb').exists()]

for k in (1, 2, 3):
    j = jaccard_local(train_pids, TRAIN_PDB_SUB, label_map,
                      lambda pid, kk=k: p2rank_topk_residues(P2RANK_TRAIN, pid, kk))
    print(f'  P2Rank top-{k} pockets:  Jaccard = {j:.4f}')

## 2  Fpocket (geometric alpha-sphere clustering)

Fpocket has no learned weights — it detects cavities purely from atom
coordinates by clustering Voronoi alpha-spheres. Install:

```bash
conda install -c conda-forge fpocket
```

Reference: Le Guilloux *et al.*, *BMC Bioinformatics* (2009).

In [ ]:
FPOCKET_BIN = shutil.which('fpocket') or '/opt/anaconda3/bin/fpocket'
assert os.path.exists(FPOCKET_BIN), 'install fpocket: `conda install -c conda-forge fpocket`'
print('fpocket binary:', FPOCKET_BIN)

In [ ]:
def run_fpocket(pdb_path, out_root, bin_path):
    base = Path(pdb_path).stem
    pid  = base.split('_')[0]
    work = Path(out_root) / pid
    work.mkdir(parents=True, exist_ok=True)
    work_pdb = work / f'{base}.pdb'
    shutil.copy(pdb_path, work_pdb)
    try:
        subprocess.run([bin_path, '-f', str(work_pdb)],
                       capture_output=True, timeout=240)
    except subprocess.TimeoutExpired:
        pass
    out_dir = work / f'{base}_out'
    if out_dir.exists():
        for f in out_dir.iterdir():
            if f.suffix in {'.pml', '.tcl', '.sh'}:
                f.unlink(missing_ok=True)
    work_pdb.unlink(missing_ok=True)
    return pid


def run_fpocket_all(pdb_root, out_root, n_expected, workers=8):
    out_root = Path(out_root); out_root.mkdir(parents=True, exist_ok=True)
    have = sum(1 for p in out_root.iterdir() if p.is_dir())
    if have >= n_expected:
        print(f'  {out_root.name}: already have {have} runs, skipping')
        return
    pdbs = sorted(Path(pdb_root).glob('*_protein.pdb'),
                  key=lambda p: int(p.stem.split('_')[0]))
    with ProcessPoolExecutor(max_workers=workers) as ex:
        futs = [ex.submit(run_fpocket, str(p), str(out_root), FPOCKET_BIN)
                for p in pdbs]
        for i, fut in enumerate(as_completed(futs), 1):
            fut.result()
            if i % 300 == 0:
                print(f'    fpocket {i}/{len(pdbs)} done')

FPOCKET_TEST  = OUT / 'fpocket_full'
FPOCKET_TRAIN = OUT / 'fpocket_train_eval'

run_fpocket_all(TEST_PDB,      FPOCKET_TEST,  N_TEST)
run_fpocket_all(TRAIN_PDB_SUB, FPOCKET_TRAIN, N_TRAIN_EVAL)

In [ ]:
def parse_pocket_pdb(path):
    seen, out = set(), []
    with open(path) as f:
        for line in f:
            if not line.startswith('ATOM'):
                continue
            chain = line[21:22]
            try: num = int(line[22:26])
            except ValueError: continue
            ic = line[26:27].strip()
            tok = f'{chain}_{num}_{ic}' if ic else f'{chain}_{num}'
            if tok not in seen:
                seen.add(tok); out.append(tok)
    return out


def fpocket_topk_residues(fpocket_dir, pid, top_k=1):
    d = Path(fpocket_dir) / str(pid) / f'{pid}_protein_out' / 'pockets'
    if not d.exists(): return set()
    out = set()
    for k in range(1, top_k + 1):
        p = d / f'pocket{k}_atm.pdb'
        if not p.exists(): break
        for tok in parse_pocket_pdb(p):
            out.add(tok)
    return out


def write_fpocket_submission(fpocket_dir, n_proteins, top_k, out_csv):
    rows = []
    for pid in range(n_proteins):
        toks = fpocket_topk_residues(fpocket_dir, pid, top_k=top_k)
        rows.append({'id': pid, 'prediction': ' '.join(sorted(toks))})
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    return out_csv

for k in (1, 3):
    p = write_fpocket_submission(FPOCKET_TEST, N_TEST, k,
                                  OUT / f'submission_fpocket_top{k}.csv')
    check_submission(p)

for k in (1, 2, 3, 5):
    j = jaccard_local(train_pids, TRAIN_PDB_SUB, label_map,
                      lambda pid, kk=k: fpocket_topk_residues(FPOCKET_TRAIN, pid, kk))
    print(f'  Fpocket top-{k} pockets:  Jaccard = {j:.4f}')

## 3  DeepSite (PlayMolecule web service)

DeepSite (Jiménez *et al.*, *Bioinformatics* 2017) is a 3D-CNN that scores
voxelised protein densities for the probability of being a binding-site
centre. The authors only distribute it as a hosted service
(`https://playmolecule.org/`), not as a downloadable model, so the cell
below is a thin REST client. It needs a registered API key and one queue
slot per protein, which makes batch use over 3 000 PDBs impractical.

We keep the code as a reference template; the submission is not produced
from DeepSite output.

In [ ]:
import requests

PLAYMOL_BASE = 'https://playmolecule.org/api'
API_KEY = os.environ.get('PLAYMOL_API_KEY', '')

def submit_deepsite(pdb_path):
    if not API_KEY: raise RuntimeError('set PLAYMOL_API_KEY first')
    files = {'pdb': open(pdb_path, 'rb')}
    headers = {'Authorization': f'Token {API_KEY}'}
    r = requests.post(f'{PLAYMOL_BASE}/apps/deepsite/run',
                      files=files, headers=headers, timeout=60)
    r.raise_for_status()
    return r.json()['job_id']

def fetch_deepsite(job_id):
    headers = {'Authorization': f'Token {API_KEY}'}
    while True:
        r = requests.get(f'{PLAYMOL_BASE}/jobs/{job_id}',
                         headers=headers, timeout=30)
        r.raise_for_status()
        data = r.json()
        if data['status'] in {'done', 'finished'}: return data['result']['residue_ids']
        if data['status'] in {'failed', 'error'}:  return []
        time.sleep(10)

print('DeepSite uses a REST queue — see PlayMolecule docs for batch usage.')

## 4  DeepSurf (3D-CNN over surface point clouds)

DeepSurf (Mylonas *et al.*, *Bioinformatics* 2021) builds a Solvent-
Accessible Surface point cloud, voxelises a local box around every surface
point and runs a ResNet-style 3D-CNN with pre-trained weights to produce a
per-point score, which is then clustered into pockets.

Public repository: <https://github.com/stemylonas/DeepSurf>. Install (own
TensorFlow env, ideally GPU):

```bash
git clone https://github.com/stemylonas/DeepSurf.git approach2/tools/DeepSurf
conda create -n deepsurf python=3.8 tensorflow=2.7 -c conda-forge -y
conda activate deepsurf
pip install -r approach2/tools/DeepSurf/requirements.txt
```

DeepSurf's `predict.py` is invoked once per PDB. Without the weights it
will not run, so the submission step is gated.

In [ ]:
DEEPSURF_DIR    = TOOLS / 'DeepSurf'
DEEPSURF_WEIGHTS = DEEPSURF_DIR / 'weights' / 'resnet_18.h5'

def run_deepsurf(pdb_path, out_dir):
    subprocess.run([sys.executable, str(DEEPSURF_DIR / 'predict.py'),
                    '-p', str(pdb_path),
                    '-mp', str(DEEPSURF_WEIGHTS),
                    '-o', str(out_dir)], check=False)

def parse_deepsurf(out_dir):
    p = Path(out_dir) / 'binding_site.csv'
    if not p.exists(): return ''
    df = pd.read_csv(p)
    sel = df[df.get('score', 0) > 0.5]
    return ' '.join(f"{r.chain}_{r.residue_id}" for r in sel.itertuples())

if DEEPSURF_DIR.exists() and DEEPSURF_WEIGHTS.exists():
    deepsurf_out = OUT / 'deepsurf_full'
    rows = []
    for pid in range(N_TEST):
        pdb = TEST_PDB / f'{pid}_protein.pdb'
        outd = deepsurf_out / str(pid); outd.mkdir(parents=True, exist_ok=True)
        run_deepsurf(pdb, outd)
        rows.append({'id': pid, 'prediction': parse_deepsurf(outd)})
    out_csv = OUT / 'submission_deepsurf.csv'
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    check_submission(out_csv)
else:
    print('DeepSurf is not installed — clone the repo and place '
          'resnet_18.h5 under tools/DeepSurf/weights/ to enable.')

## 5  Custom 3D voxel U-Net (reference)

A compact 3D segmentation network that voxelises each protein into a
`64 × 64 × 64 × 4` density grid (one channel per element type) and
predicts a per-voxel binding probability. Inference would average the
predicted probability inside a 4 Å ball centred on every Cα atom; residues
above a tuned threshold become the submission tokens.

Training on the 16 379 PDBs in `train.zip` takes hours on CPU and is not
executed here — we only ship the voxelisation routine and the network
definition.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

GRID_RES = 1.0
BOX      = 64

def voxelise(coords, elements, centre=None):
    coords = np.asarray(coords)
    if centre is None:
        centre = coords.mean(axis=0)
    origin = centre - BOX * GRID_RES / 2
    grid = np.zeros((4, BOX, BOX, BOX), dtype=np.float32)
    ch = {'C': 0, 'N': 1, 'O': 2, 'S': 3, 'P': 3}
    for (x, y, z), el in zip(coords, elements):
        c = ch.get(el)
        if c is None: continue
        gx = int((x - origin[0]) / GRID_RES)
        gy = int((y - origin[1]) / GRID_RES)
        gz = int((z - origin[2]) / GRID_RES)
        if 0 <= gx < BOX and 0 <= gy < BOX and 0 <= gz < BOX:
            grid[c, gx, gy, gz] += 1.0
    return grid, origin

class VoxelUNet(nn.Module):
    def __init__(self, in_ch=4, base=16):
        super().__init__()
        self.enc1 = self._block(in_ch, base)
        self.enc2 = self._block(base, base * 2)
        self.enc3 = self._block(base * 2, base * 4)
        self.bot  = self._block(base * 4, base * 8)
        self.up3  = nn.ConvTranspose3d(base * 8, base * 4, 2, stride=2)
        self.dec3 = self._block(base * 8, base * 4)
        self.up2  = nn.ConvTranspose3d(base * 4, base * 2, 2, stride=2)
        self.dec2 = self._block(base * 4, base * 2)
        self.up1  = nn.ConvTranspose3d(base * 2, base, 2, stride=2)
        self.dec1 = self._block(base * 2, base)
        self.out  = nn.Conv3d(base, 1, 1)
    def _block(self, ic, oc):
        return nn.Sequential(
            nn.Conv3d(ic, oc, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv3d(oc, oc, 3, padding=1), nn.ReLU(inplace=True))
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(F.max_pool3d(e1, 2))
        e3 = self.enc3(F.max_pool3d(e2, 2))
        b  = self.bot(F.max_pool3d(e3, 2))
        d3 = self.dec3(torch.cat([self.up3(b), e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        return torch.sigmoid(self.out(d1))

model = VoxelUNet()
print('parameters:', sum(p.numel() for p in model.parameters()))

## 6  Final submission (P2Rank top-2)

Of the methods that actually ran end-to-end (P2Rank, Fpocket) the best
local Jaccard is **P2Rank top-2 pockets** at 0.2816 on 200 train proteins.
Fpocket peaks at 0.182, ensembles (union / intersection) do not improve.
The final file shipped to the leaderboard is therefore the P2Rank top-2
submission.

In [ ]:
FINAL = APPROACH / 'submission.csv'
shutil.copy(OUT / 'submission_p2rank_top2.csv', FINAL)
check_submission(FINAL)
pd.read_csv(FINAL).head(5)

## 7  Sanity check against `metric.py`

Build a fake target frame for the train subset, feed both files through the
exact `parse_submission` / `jaccard_score` pipeline of `metric.py`, and
confirm we get the same number printed in section 6.

In [ ]:
target_rows = []
sub_rows    = []
for pid in train_pids:
    univ  = residue_universe(TRAIN_PDB_SUB / f'{pid}_protein.pdb')
    truth = label_map.get(pid, set())
    for tok in univ:
        target_rows.append({'id': pid, 'residue_id': tok,
                            'true': 1 if tok in truth else 0})
    sub_rows.append({'id': pid,
                     'prediction': ' '.join(sorted(p2rank_topk_residues(P2RANK_TRAIN, pid, 2)))})

tgt_df = pd.DataFrame(target_rows)
sub_df = parse_submission(pd.DataFrame(sub_rows))
merged = pd.merge(tgt_df, sub_df, on=['id', 'residue_id'], how='left')
merged['prediction'] = merged['prediction'].fillna(0).astype(int)
print(f'  P2Rank top-2 vs train labels (metric.py replication): '
      f'Jaccard = {jaccard_score(merged.true, merged.prediction):.4f}')

## Summary

| method      | runs locally       | pre-trained          | per-residue | local Jaccard | used   |
| ----------- | ------------------ | -------------------- | ----------- | ------------- | ------ |
| P2Rank      | yes (Java)         | yes (`default.model`) | yes         | **0.282**     | **yes**|
| Fpocket     | yes                | n/a (geometric)      | indirect    | 0.182         | no     |
| DeepSite    | web API only       | yes (server-side)    | indirect    | —             | no     |
| DeepSurf    | yes (GPU recommended) | yes (`resnet_18.h5`) | yes      | —             | no     |
| 3D U-Net    | training required  | no                   | yes         | —             | no     |

Final file: `02_offshelf_tools/submission.csv` (P2Rank top-2 pockets), which
scored **0.268** on the public leaderboard.